In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# 2. 작업하던 원래 폴더로 이동
%cd '/content/drive/MyDrive/[이스트캠프]AI 휴먼/10.할래말래 프로젝트/강윤찬'

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1AUFjYAj7V56mZjIPWI7OowAMZyxQw0Wi/[이스트캠프]AI 휴먼/10.할래말래 프로젝트/강윤찬


In [ ]:
# Unsloth 및 필수 패키지 설치
# 빠듯한 일정과 무료 컴퓨팅 자원(Colab T4 GPU)의 한계를 극복하기 위해, 기존 Hugging Face 방식보다 메모리 사용량을 60% 이상 줄이고 속도는 2배 빠른 Unsloth 라이브러리를 사용하겠습니다.

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-glx8q5dk/unsloth_4a73b6f2393d4418bc226cdfa0b72d7d
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-glx8q5dk/unsloth_4a73b6f2393d4418bc226cdfa0b72d7d
  Resolved https://github.com/unslothai/unsloth.git to commit 997f1a7ce5fb7a0319c2b8abe0e7af02e2160efe
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 21.9 MB/s eta 0:00:0

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512 # VRAM 절약을 위해 시퀀스 길이 제한
dtype = None # 자동 감지
load_in_4bit = True # 4비트 양자화 (T4 GPU 필수)

# CJK tokenizer가 강한 Qwen모델로 변경
print("1. Qwen3 4B 모델 로드 중...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
1. Qwen3 4B 모델 로드 중...
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [ ]:
print("2. LoRA 어댑터 적용 중...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("모델 세팅 완료!")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


2. LoRA 어댑터 적용 중...


Unsloth 2026.3.4 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


모델 세팅 완료!


In [ ]:
import json
from datasets import Dataset

# 1. 업로드한 데이터셋 불러오기
with open("./datas/fact_coach_dataset_final_v3.json", "r", encoding="utf-8") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

In [ ]:
# output-only loss를 통해 instruction은 학습하지 않고
# output만을 학습하여 일반화를 높이고 불필요한 패턴 학습을 낮춘다.

def format_and_mask(examples):
    texts, prompt_lens = [], []
    for instruction, output in zip(examples["instruction"], examples["output"]):
        prompt = f"""당신은 30대 K-직장인의 핑계를 무자비하게 박살 내는 극대노 팩트폭행 헬스 코치입니다.
        항상 자연스러운 한국어로만 답하세요.
        한자나 영어 단어를 섞지 마세요.
        한국인이 대화하는 것처럼 자연스럽게 말하세요.

### 사용자 핑계:
{instruction}

### 코치의 팩트폭행:
"""
        full_text = prompt + output + tokenizer.eos_token
        texts.append(full_text)
        prompt_lens.append(len(tokenizer(prompt)["input_ids"]))
    return {"text": texts, "prompt_len": prompt_lens}

formatted_dataset = dataset.map(format_and_mask, batched=True)

# tokenize + output-only labels
def tokenize_and_mask(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_seq_length,
    )
    labels = tokenized["input_ids"].copy()
    prompt_len = example["prompt_len"]
    labels[:prompt_len] = [-100] * prompt_len  # output-only loss
    tokenized["labels"] = labels
    return tokenized

tokenized_dataset = formatted_dataset.map(
    tokenize_and_mask,
    remove_columns=formatted_dataset.column_names
)

print(f"총 {len(formatted_dataset)}개의 데이터 학습 준비 완료!")
print("\n[데이터 샘플 확인]")
print(formatted_dataset[0]['text'])

Map:   0%|          | 0/415 [00:00<?, ? examples/s]

Map:   0%|          | 0/415 [00:00<?, ? examples/s]

총 415개의 데이터 학습 준비 완료!

[데이터 샘플 확인]
당신은 30대 K-직장인의 핑계를 무자비하게 박살 내는 극대노 팩트폭행 헬스 코치입니다.
        항상 자연스러운 한국어로만 답하세요.
        한자나 영어 단어를 섞지 마세요.
        한국인이 대화하는 것처럼 자연스럽게 말하세요.

### 사용자 핑계:
대체 나한테 왜 이런 일이 생기는 거지?

### 코치의 팩트폭행:
네 인생에 무슨 일이라도 일어났다고 생각하냐? 😒 인슐린 저항성 때문에 체지방률만 뻥튀기면서 살고 있을 뿐인데 무슨 큰 일이라도 난 것처럼 우물우물하냐? 🤦‍♂️ 당장 코르티솔 수치를 낮추는 게 먼저다. 🔔
1. 당장 물 1리터 마시고 혈액순환 돌리기.
2. 의자에 앉아서 허리 펴고 등 근력 운동 3세트 하기.
3. 일과 마무리 후 10분간 명상하기. 🙏🌱💆‍♀️<|im_end|>


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = tokenized_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 4,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 파인튜닝을 시작합니다...")
trainer_stats = trainer.train()
print("✅ 파인튜닝 완료!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 파인튜닝을 시작합니다...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 415 | Num Epochs = 4 | Total steps = 208
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Step,Training Loss
10,7.178082
20,5.338210
30,5.211053
40,4.987565
50,5.115429
60,4.969698
70,4.959448
80,5.063976
90,4.874317
100,4.955796


✅ 파인튜닝 완료!


In [ ]:
# 구글 드라이브에 저장하거나 Colab 로컬 폴더에 저장
model.save_pretrained("./models/fact_coach_lora_model_qwen4")
tokenizer.save_pretrained("./models/fact_coach_lora_model_qwen4")
print("✅ 모델 저장 완료! 좌측 폴더 탭에서 'fact_coach_lora_model' 폴더를 확인할 수 있습니다.")

✅ 모델 저장 완료! 좌측 폴더 탭에서 'fact_coach_lora_model' 폴더를 확인할 수 있습니다.
